# TinyHybridNet on Tiny-ImageNet — FP32 & QAT (INT8)

**Dataset:** Tiny-ImageNet-200 (200 classes, 64×64 RGB)
**Goal:** Train `TinyHybridNet` (Fire-inspired mobile-residual CNN) in FP32,
then apply Quantization-Aware Training (QAT) to obtain an INT8 version.
Report **Top-1** and **Top-5** accuracy for both.

## Notebook layout

1. Configuration & reproducibility
2. Dataset & loaders
3. Model definition (`FireMobileResidual`, `TinyHybridNet`) + auto fuse-map
4. QAT helpers (fuse + prepare)
5. FP32 training
6. QAT training and INT8 conversion
7. Evaluation — Top-1 and Top-5 — for FP32 and INT8
8. Final comparison table & ranking
9. Persist experiment summary
10. (Optional) TorchScript export of the INT8 model

## 1. Configuration & reproducibility

In [ ]:
# Standard library
import json
import os
import random
import sys
from dataclasses import asdict, replace
from pathlib import Path

# Third-party
import numpy as np
import torch
import torch.nn as nn
import torchinfo

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Local — ml package
from ml import (
    DataConfig, TrainerConfig, QATConfig,
    create_imagenet_loaders,
    MODEL_REGISTRY, register_model,
    Trainer, make_qat_callback,
    find_fuse_groups,
    load_best_model, convert_to_int8,
    auto_resume_path,
    create_results_summary, disk_mb,
    compute_flops, make_run_summary,
)
from configs.loader import load_config

# Model architectures
from models import (
    FireMobileResidual, TinyHybridNet, InvertedResidual, TinyMobileNetV2, )

torch.backends.quantized.engine = "fbgemm"

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_DIR = project_root / "checkpoints" / "tinyhybridnet_qat"
RESULTS_DIR = project_root / "results" / "tinyhybridnet_qat"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
for _sub in ("models", "tables", "summaries"):
    (RESULTS_DIR / _sub).mkdir(parents=True, exist_ok=True)

# ─── Hyperparameters — loaded from configs/, override here if needed ──────────
data_cfg = DataConfig(**load_config("data.yaml"))
data_cfg.seed = SEED  # tie dataset split seed to notebook-level SEED constant
fp32_cfg = TrainerConfig(**load_config("training.yaml"))
qat_cfg = QATConfig(**load_config("qat.yaml"))
# ─────────────────────────────────────────────────────────────────────────────
print(device)

## 2. Dataset & loaders

We use Tiny-ImageNet-200 from Kaggle. The train folder is split 90/10 into
train/val with a **seeded generator** so the split is identical on every run.

* Training transforms: light augmentation (random crop + hflip + color jitter)
* Validation transforms: deterministic resize + center crop

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")
data_cfg.dataset_path = train_path
print("Tiny-ImageNet train path:", train_path)

train_ds, val_ds, train_loader, val_loader = create_imagenet_loaders(data_cfg)

print(f"Train samples: {len(train_ds):,}")
print(f"Val   samples: {len(val_ds):,}")
print(f"Classes:       {data_cfg.num_classes}")
print(f"Batches/epoch: train={len(train_loader)}, val={len(val_loader)}")

## 3. Model architecture — TinyHybridNet

* **Stem**: 3 → 32 channels (3×3 conv + BN + ReLU)
* **FireMobileResidual blocks**: 1×1 squeeze → 3×3 depthwise → 1×1 expand,
  with a residual add (`FloatFunctional`, quantization-friendly) and a final
  ReLU applied after the add.
* **Head**: `AdaptiveAvgPool2d(1)` → `Linear(256, num_classes)`

### Fuse map — built automatically, not hand-written

Unlike the AlexNet notebook (a flat `Sequential` where `[conv_idx, relu_idx]`
pairs can just be read off), `TinyHybridNet` nests `Conv-BN-ReLU` groups
several levels deep — inside `FireMobileResidual.block` and `.shortcut`.
Hand-writing those module paths is error-prone, so `find_fuse_groups` walks
the **entire** module tree recursively (not just top-level `Sequential`
containers) and detects `Conv2d → BatchNorm2d (→ ReLU)` runs wherever they
occur, fusing `[conv, bn, relu]` where a ReLU follows, and `[conv, bn]`
otherwise. This picks up: the stem (1 group), each block's two
depthwise-separable conv-bn-relu pairs plus its non-ReLU expansion conv-bn
(3 groups/block × 6 blocks), and the conv-bn projection inside any
`shortcut` (4 of the 6 blocks change channels/stride, so 4 more groups) —
22 groups total. The residual-add `ReLU` (applied *after* `skip_add`,
outside `block`) is correctly left unfused, since fusing across the
residual connection isn't valid.

In [ ]:
# FireMobileResidual, TinyHybridNet imported from models/baselines.py
# find_fuse_groups imported from ml.quantization

MODEL_REGISTRY.clear()
register_model(
    "tinyhybridnet",
    fuse_map=find_fuse_groups(build_tinyhybridnet()),
    lr=3e-4,
)

print(f"Fuse map ({len(MODEL_REGISTRY['tinyhybridnet']['fuse_map'])} groups):")
for g in MODEL_REGISTRY["tinyhybridnet"]["fuse_map"]:
    print(" ", g)

# Smoke test
x = torch.randn(2, 3, data_cfg.img_size, data_cfg.img_size)
m = build_tinyhybridnet().eval()
with torch.no_grad():
    y = m(x)
assert y.shape == (2, data_cfg.num_classes)
info = torchinfo.summary(m, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
print(f"TinyHybridNet OK -> {tuple(y.shape)}, params={info.total_params/1e6:.2f}M")

### 3b. Model architecture — TinyMobileNetV2 (inverted residual)

TinyHybridNet uses a squeeze-first Fire block (1×1 squeeze → depthwise →

1×1 expand). TinyMobileNetV2 flips that order — the classic MobileNetV2

inverted residual: 1×1 expand → 3×3 depthwise → 1×1 project, with

the residual applied only when shapes match (no stride/channel change).

This block has no internal squeeze step; "skinny" feature maps live at the

block's input/output, and the expanded (wide) tensor only exists briefly

inside the block — the opposite memory/compute profile from Fire blocks.

Same building blocks otherwise (FloatFunctional residual add, BN, ReLU),

so it reuses find_fuse_groups unchanged.

In [ ]:
# InvertedResidual, TinyMobileNetV2 imported from models/efficient_cnns.py

register_model(
    "tinymobilenetv2",
    fuse_map=find_fuse_groups(build_tinymobilenetv2()),
    lr=3e-4,
)

print(f"Fuse map ({len(MODEL_REGISTRY['tinymobilenetv2']['fuse_map'])} groups):")
for g in MODEL_REGISTRY["tinymobilenetv2"]["fuse_map"]:
    print(" ", g)

x = torch.randn(2, 3, data_cfg.img_size, data_cfg.img_size)
m = build_tinymobilenetv2().eval()
with torch.no_grad():
    y = m(x)
assert y.shape == (2, data_cfg.num_classes)
info = torchinfo.summary(m, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
print(f"TinyMobileNetV2 OK -> {tuple(y.shape)}, params={info.total_params/1e6:.2f}M")

## 5. FP32 training

In [ ]:
fp32_models = {}
fp32_training_results = {}

for name, spec in MODEL_REGISTRY.items():
    model_cfg = replace(fp32_cfg, lr=spec["lr"]) if "lr" in spec else fp32_cfg
    print("=" * 72)
    print(f"Training: {name}  lr={model_cfg.lr}  epochs={model_cfg.epochs}")
    print("=" * 72)

    model = spec["ctor"]().to(device)

    trainer = Trainer(
        model, train_loader, val_loader, model_cfg,
        device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
    )
    results = trainer.fit()

    fp32_training_results[name] = results
    fp32_models[name] = model.cpu()

    del model
    torch.cuda.empty_cache()

print("\nFP32 training complete.")

In [ ]:
# FP32 training with auto-resume support
fp32_models = {}
fp32_training_results = {}

for name, spec in MODEL_REGISTRY.items():
    # Auto-detect resume checkpoint
    resume_from = auto_resume_path(SAVE_DIR, name)
    
    # If best already exists and no resume checkpoint, skip
    if (SAVE_DIR / f"{name}_best.pth").exists() and not resume_from:
        print(f"Skipping {name} (already trained) — reloading checkpoint.")
        fp32_models[name] = load_best_model(name, spec["ctor"], SAVE_DIR, device="cpu")
        continue
    
    # Restore W&B run if resuming
    run_id = None
    if resume_from:
        meta_path = SAVE_DIR / f"{name}_meta.json"
        if meta_path.exists():
            run_id = json.loads(meta_path.read_text()).get("wandb_run_id")

    model_cfg = replace(fp32_cfg, lr=spec["lr"]) if "lr" in spec else fp32_cfg
    print("=" * 72)
    print(f"Training: {name}  lr={model_cfg.lr}  epochs={model_cfg.epochs}")
    if resume_from:
        print(f"(Resuming from checkpoint)")
    print("=" * 72)

    model = spec["ctor"]().to(device)

    trainer = Trainer(
        model, train_loader, val_loader, model_cfg,
        device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
    )
    results = trainer.fit(resume_from=resume_from)

    fp32_training_results[name] = results
    fp32_models[name] = model.cpu()

    del model
    torch.cuda.empty_cache()

print("\nFP32 training complete.")

## 6. Quantization-Aware Training (QAT)

QAT fine-tuning happens on GPU; `convert()` and INT8 inference must run on
CPU. We use an `epoch_callback` (passed straight to `train_model`) to freeze
BatchNorm running stats and disable the fake-quant observers in the later
epochs, per `config["qat_freeze_bn_epoch"]` / `config["qat_disable_observer_epoch"]`.

> Note: results are stored under the **plain architecture name**
> (`qat_training_results[name]`), matching how the AlexNet notebook's
> `qat_training_results` dict is keyed — `f"qat_{name}"` is only used for
> the checkpoint filename prefix, not the dict key. Any downstream analysis
> script reading `experiment_summary.json` should look up QAT results by
> the plain name, not `"qat_" + name`.

In [ ]:
qat_train_cfg = replace(
    fp32_cfg,
    epochs=qat_cfg.epochs,
    lr=qat_cfg.lr,
    weight_decay=qat_cfg.weight_decay,
    use_amp=False,
)

qat_models = {}
qat_training_results = {}

for name in MODEL_REGISTRY:
    # Auto-detect resume checkpoint
    resume_from = auto_resume_path(SAVE_DIR, f"qat_{name}")
    
    # Restore W&B run if resuming
    run_id = None
    if resume_from:
        meta_path = SAVE_DIR / f"qat_{name}_meta.json"
        if meta_path.exists():
            run_id = json.loads(meta_path.read_text()).get("wandb_run_id")

    print("=" * 72)
    print(f"QAT fine-tuning: {name}")
    if resume_from:
        print(f"(Resuming from checkpoint)")
    print("=" * 72)

    qat_model = build_qat(name, save_dir=SAVE_DIR, device=device)
    cb = make_qat_callback(qat_cfg.freeze_bn_epoch, qat_cfg.disable_observer_epoch)

    trainer = Trainer(
        qat_model, train_loader, val_loader, qat_train_cfg,
        device, SAVE_DIR, f"qat_{name}", num_classes=data_cfg.num_classes,
        epoch_callback=cb,
    )
    results = trainer.fit(resume_from=resume_from)

    qat_training_results[name] = results
    qat_models[name] = qat_model.cpu()

    del qat_model
    torch.cuda.empty_cache()

print("\nQAT training complete.")

## 7. INT8 conversion and CPU evaluation

`tq.convert` materialises true quantised ops; this must run on CPU with the
model in `eval()` mode. We build a small CPU-side val loader
(`num_workers=0`) and reuse `evaluate_topk` for Top-1/Top-5.

In [ ]:
val_loader_cpu = torch.utils.data.DataLoader(
    val_ds, batch_size=data_cfg.batch_size, shuffle=False, num_workers=0, pin_memory=False,
)
cpu_device = torch.device("cpu")

int8_models = {}
for name in MODEL_REGISTRY:
    int8_models[name] = convert_to_int8(qat_models[name].cpu().eval())

for name, m in int8_models.items():
    torch.save(m.state_dict(), RESULTS_DIR / "models" / f"{name}.pth")
print("INT8 conversion done.")

int8_metrics = {}
dummy_cfg = replace(fp32_cfg, use_amp=False)
for name, m in int8_models.items():
    m = m.cpu().eval()
    trainer = Trainer(m, val_loader_cpu, val_loader_cpu, dummy_cfg, cpu_device, SAVE_DIR, name,
                      num_classes=data_cfg.num_classes)
    metrics = trainer.evaluate(topk=(1, 5))
    int8_metrics[name] = metrics
    print(f"{name:22s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")

## 8. FP32 evaluation — reload best checkpoints

Reload the best FP32 checkpoint from disk to make sure we're evaluating the
*best* epoch, not whatever was in memory at the end of training.

In [ ]:
fp32_models = {
    name: load_best_model(name, spec["ctor"], SAVE_DIR, device)
    for name, spec in MODEL_REGISTRY.items()
}

fp32_metrics = {}
dummy_cfg = replace(fp32_cfg, use_amp=False)
for name, model in fp32_models.items():
    trainer = Trainer(model, train_loader, val_loader, dummy_cfg, device, SAVE_DIR, name,
                      num_classes=data_cfg.num_classes)
    metrics = trainer.evaluate(topk=(1, 5))
    fp32_metrics[name] = metrics
    print(f"{name:22s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")

## 9. Final comparison — Top-1, Top-5, size, params

* **Top-1 / Top-5 accuracy** on the validation split
* **Trainable parameters** (millions)
* **Disk size** of the checkpoint (MB) — for INT8 this is the *real*
  on-disk size

In [ ]:
rows = []

for name, m in fp32_models.items():
    info = torchinfo.summary(m, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    rows.append({
        "model": name, "precision": "FP32",
        "top1_%": fp32_metrics[name]["top1"], "top5_%": fp32_metrics[name]["top5"],
        "loss": fp32_metrics[name]["loss"],
        "params_M": info.total_params / 1e6,
        "size_MB": disk_mb(SAVE_DIR / f"{name}_best.pth"),
    })

for name, m in int8_models.items():
    fp32_info = torchinfo.summary(fp32_models[name], input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    rows.append({
        "model": f"{name}_INT8", "precision": "INT8",
        "top1_%": int8_metrics[name]["top1"], "top5_%": int8_metrics[name]["top5"],
        "loss": int8_metrics[name]["loss"],
        "params_M": fp32_info.total_params / 1e6,
        "size_MB": disk_mb(RESULTS_DIR / "models" / f"{name}.pth"),
    })

df = build_comparison_table(rows)
df.to_csv(RESULTS_DIR / "tables" / "final_comparison.csv", index=False)

print("=" * 72)
print("RANKING BY TOP-1 ACCURACY (all precisions)")
print("=" * 72)
ranked = df.sort_values("top1_%", ascending=False).reset_index(drop=True)
for i, row in ranked.iterrows():
    print(f"{i+1:2d}. {row['model']:22s} [{row['precision']}] | "
          f"top1={row['top1_%']:6.2f}% | top5={row['top5_%']:6.2f}% | "
          f"params={row['params_M']:6.2f}M | size={row['size_MB']:6.2f}MB")

In [ ]:
import json as _json

# --- Benchmark FP32 models (GPU) ---
fp32_benchmarks = {}
for name, m in fp32_models.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    t = Trainer(m, train_loader, val_loader, dummy_cfg, device, SAVE_DIR, name,
                num_classes=data_cfg.num_classes)
    fp32_benchmarks[name] = t.benchmark()
    print(f"{name:22s} FP32 | {fp32_benchmarks[name]['latency_ms_per_image']:.3f} ms/img")

# --- Benchmark INT8 models (CPU) ---
int8_benchmarks = {}
for name, m in int8_models.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    t = Trainer(m.cpu().eval(), val_loader_cpu, val_loader_cpu, dummy_cfg, cpu_device, SAVE_DIR, name,
                num_classes=data_cfg.num_classes)
    int8_benchmarks[name] = t.benchmark(warmup=50)
    print(f"{name:22s} INT8 | {int8_benchmarks[name]['latency_ms_per_image']:.3f} ms/img")

# --- FLOPs ---
fp32_flops = {}
for name, m in fp32_models.items():
    fp32_flops[name] = compute_flops(m.cpu().eval())
    print(f"{name:22s} | {fp32_flops[name]['macs']/1e9:.3f} GMACs")

# --- Per-model summary JSONs ---
(RESULTS_DIR / "summaries").mkdir(parents=True, exist_ok=True)
for name in fp32_models:
    info = torchinfo.summary(fp32_models[name], input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    summary = make_run_summary(
        name=name,
        mode="fp32+qat+int8",
        fit_results=fp32_training_results.get(name, {}),
        fp32_eval=fp32_metrics[name],
        params_m=info.total_params / 1e6,
        fp32_size_mb=disk_mb(SAVE_DIR / f"{name}_best.pth"),
        int8_size_mb=disk_mb(RESULTS_DIR / "models" / f"{name}.pth"),
        fp32_benchmark=fp32_benchmarks[name],
        flops_results=fp32_flops[name],
        int8_eval=int8_metrics.get(name),
        int8_benchmark=int8_benchmarks.get(name),
    )
    out = RESULTS_DIR / "summaries" / f"{name}_summary.json"
    with open(out, "w") as f:
        _json.dump(summary, f, indent=2, default=str)
    print(f"Saved: {out.name}")

## 10. Persist experiment summary

A single JSON next to the checkpoints captures: config, system info, and
per-model metrics. Re-running this notebook with the same seed should
reproduce these numbers up to non-determinism in cuDNN kernels.

In [ ]:
create_results_summary(
    results={
        "fp32_metrics": fp32_metrics,
        "int8_metrics": int8_metrics,
        "fp32_training_results": fp32_training_results,
        "qat_training_results": qat_training_results,
    },
    config=asdict(fp32_cfg),
    output_path=RESULTS_DIR / "summaries" / "experiment_summary.json",
)
print(f"Summary written to: {RESULTS_DIR / 'summaries' / 'experiment_summary.json'}")

## 11. (Optional) TorchScript export of the INT8 model

Quantized models with grouped/depthwise convs can be brittle under
`torch.jit.script`, so tracing with a representative input is the more
reliable export path here.

In [ ]:
example_input = torch.randn(1, 3, data_cfg.img_size, data_cfg.img_size)

for name, m in int8_models.items():
    m = m.cpu().eval()
    traced = torch.jit.trace(m, example_input)
    ts_path = RESULTS_DIR / "models" / f"{name}_int8_ts.pt"
    traced.save(str(ts_path))
    print(f"Saved TorchScript INT8 model: {ts_path}")